In [ ]:
import numpy as np
import pandas as pd
import time

# الحد الأقصى لعدد الحالات لتجنب استهلاك الذاكرة (1 مليون حالة)
MAX_CASES = 1_000_000


In [ ]:
def generate_dataset(n_inputs=10, gate_type=1):
    total_cases = 2 ** n_inputs

    if total_cases > MAX_CASES:
        raise ValueError(f"Number of cases ({total_cases}) exceeds MAX_CASES ({MAX_CASES})")

    X, y = [], []

    for i in range(total_cases):
        binary = format(i, f'0{n_inputs}b')
        row = [int(b) for b in binary]
        X.append(row)

        if gate_type == 1:      # AND3
            output = 1 if all(row) else 0
        elif gate_type == 2:    # OR
            output = 1 if any(row) else 0
        elif gate_type == 3:    # NAND
            output = 0 if all(row) else 1
        elif gate_type == 4:    # NOR
            output = 0 if any(row) else 1
        else:
            raise ValueError("Invalid gate_type")

        y.append(output)

    return np.array(X), np.array(y)

In [ ]:
class Perceptron:
    def __init__(self, learning_rate=0.125):
        self.lr = learning_rate
        self.weights = None
        self.bias = 0.0

    def fit(self, X, y):
        n_samples, n_features = X.shape
        #self.weights = np.random.uniform(-0.01,0.01,n_features)
        self.weights = np.zeros(n_features)
        epoch = 0
        start_time = time.time()

        while True:
            total_error = 0
            epoch += 1

            # التدريب باستخدام حلقة فور (Stochastic) لجميع الحالات
            for i in range(n_samples):
                # الحساب اليدوي لكل عينة
                linear = np.dot(X[i], self.weights) + self.bias
                y_pred = 1 if linear >= 0 else 0
                
                # حساب الخطأ
                error = y[i] - y_pred
                
                if error != 0:
                    total_error += abs(error)
                    # تحديث الأوزان والانحياز فوراً عند اكتشاف خطأ
                    self.weights += self.lr * error * X[i]
                    self.bias += self.lr * error

            print(f"Epoch {epoch} | Total Error: {total_error}")

            # التوقف عند انعدام الأخطاء في الدورة الكاملة
            if total_error == 0:
                print(f"\nConverged at epoch {epoch}")
                break

        return time.time() - start_time

def predict(model, X):
    y_pred = []
    for i in range(X.shape[0]):
        linear_output = np.dot(X[i], model.weights) + model.bias
        y_pred.append(1 if linear_output >= 0 else 0)
    return np.array(y_pred)

In [ ]:
# إدخال المستخدم
try:
    n = int(input("Enter number of inputs (default = 10): ") or 10)
    gate = int(input("Enter gate type (1=AND, 2=OR, 3=NAND, 4=NOR) [default=1]: ") or 1)
except:
    n, gate = 10, 1

# التحقق من الحجم
if (2 ** n) > MAX_CASES:
    print(f"\nNumber of cases ({2**n}) exceeds limit! Using 10 inputs instead.")
    n = 10

# توليد وعرض البيانات
X, y = generate_dataset(n_inputs=n, gate_type=gate)
columns = [f"x{i+1}" for i in range(X.shape[1])] + ["y"]
df = pd.DataFrame(np.column_stack((X, y)), columns=columns)

print("\nDataset Preview (First 5 rows):")
print(df.head())

In [ ]:
model = Perceptron(learning_rate=0.125)

print("\nStarting Training...")
execution_time = model.fit(X, y)

# حساب الدقة
y_pred = predict(model, X)
accuracy = np.mean(y_pred == y)

print("-" * 30)
print(f"Final weights: {model.weights}")
print(f"Final bias: {model.bias}")
print(f"Execution time: {execution_time:.6f} seconds")
print(f"Accuracy: {accuracy * 100:.2f}%")